# 🧠 BERT Sentiment Analysis for Local PC

**Advanced Sentiment Analysis with Pretrained BERT Models**

This notebook provides a comprehensive sentiment analysis solution optimized for local PC environments with GPU support. It includes:

- 🚀 **GPU Acceleration** - Automatic GPU detection and utilization (CUDA/CPU)
- 🤖 **Multiple BERT Models** - Support for various pretrained models
- 📊 **Rich Visualizations** - Interactive charts and statistical analysis
- 💾 **Local File Support** - Easy data import from your local file system
- 🔧 **Fine-tuning Capabilities** - Advanced model customization
- 📈 **Comprehensive Analytics** - User and post-level sentiment insights

---

## 📋 Requirements

- Python 3.8+ with Jupyter Notebook/Lab
- CUDA-compatible GPU (optional, but recommended)
- CSV file with comment data
- Columns: `comment_text`, `comment_owner_username` (required)
- Optional columns: `post_id`, `post_owner_username`, `comment_likes`, `post_likes`

### 🛠️ Installation Requirements

```bash
pip install transformers[torch] datasets accelerate
pip install plotly pandas numpy tqdm scikit-learn
pip install ipywidgets matplotlib seaborn
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
```

---

In [ ]:
# 📦 Package Installation Check & Setup

# Check if required packages are installed
import subprocess
import sys

def install_package(package):
    """Install package if not already installed"""
    try:
        __import__(package.split('[')[0])
        print(f"✅ {package} is already installed")
    except ImportError:
        print(f"📥 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed successfully")

# Required packages for local environment
required_packages = [
    "transformers[torch]",
    "datasets",
    "accelerate",
    "plotly",
    "pandas",
    "numpy",
    "tqdm",
    "scikit-learn",
    "ipywidgets",
    "matplotlib",
    "seaborn"
]

print("📦 Checking and installing required packages...")
for package in required_packages:
    install_package(package)

print("\n✅ All packages are ready!")
print("💻 Local environment setup completed!")

In [ ]:
# 📚 Import Required Libraries

import pandas as pd
import numpy as np
import torch
import json
import time
import random
import warnings
import os
from datetime import datetime
from tqdm.auto import tqdm
from pathlib import Path

# Transformers and ML libraries
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    pipeline,
    Trainer, 
    TrainingArguments
)
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, confusion_matrix

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import matplotlib.pyplot as plt
import seaborn as sns

# Jupyter specific imports
from IPython.display import display, HTML, clear_output, FileLink
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

# File handling for local environment
import tkinter as tk
from tkinter import filedialog

# Suppress warnings
warnings.filterwarnings('ignore')
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Enable widget extensions for Jupyter
try:
    from IPython.display import Javascript
    display(Javascript("IPython.OutputArea.prototype._should_scroll = function(lines) { return false; }"))
except:
    pass

print("📚 All libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers available: True")
print(f"📊 Plotly available: True")
print(f"💻 Local environment ready!")

In [ ]:
# 🚀 GPU Setup and Device Configuration for Local PC

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_available = torch.cuda.is_available()

print("🖥️  Device Configuration:")
print(f"   Device: {device}")
print(f"   GPU Available: {gpu_available}")

if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_count = torch.cuda.device_count()
    print(f"   GPU Name: {gpu_name}")
    print(f"   GPU Memory: {gpu_memory:.1f} GB")
    print(f"   GPU Count: {gpu_count}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   cuDNN Available: {torch.backends.cudnn.enabled}")
    
    # Set memory fraction to avoid OOM errors
    torch.cuda.empty_cache()
    print("   ✅ GPU is ready for use!")
else:
    print("   ⚠️  GPU not available, using CPU (will be slower)")
    print("   💡 To use GPU: Install CUDA-compatible PyTorch")
    print("   💻 Install command: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

# Check if running in Jupyter
try:
    get_ipython()
    jupyter_env = True
    print(f"   📝 Running in Jupyter environment")
except NameError:
    jupyter_env = False
    print(f"   🐍 Running in Python script mode")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Create output directory
output_dir = Path("sentiment_analysis_output")
output_dir.mkdir(exist_ok=True)
print(f"\n📁 Output directory: {output_dir.absolute()}")

print("\n🎯 Environment setup completed!")

In [ ]:
# 🤖 BERT Sentiment Analyzer Class

class BERTSentimentAnalyzer:
    """Enhanced BERT Sentiment Analyzer with GPU support and fine-tuning capabilities"""
    
    def __init__(self, model_name="distilbert-base-uncased", learning_rate=2e-5, 
                 batch_size=16, epochs=3, use_fine_tuning=False):
        self.tokenizer = None
        self.model = None
        self.pipeline = None
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Hyperparameters
        self.model_name = model_name
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.use_fine_tuning = use_fine_tuning
        
        # Memory management
        self.chunk_size = 5000
        
        print(f"🤖 Initializing BERT Sentiment Analyzer")
        print(f"   Model: {model_name}")
        print(f"   Device: {self.device}")
        print(f"   Batch Size: {batch_size}")
        print(f"   Fine-tuning: {use_fine_tuning}")
    
    def load_model(self, model_name=None, use_gpu=True):
        """Load BERT model and tokenizer"""
        model_name = model_name or self.model_name
        device_id = 0 if use_gpu and torch.cuda.is_available() else -1
        
        print(f"📥 Loading model: {model_name}")
        
        try:
            # Load the sentiment analysis pipeline
            self.pipeline = pipeline(
                "sentiment-analysis",
                model=model_name,
                device=device_id,
                return_all_scores=True
            )
            
            # Test the pipeline
            test_result = self.pipeline(["This is a test sentence."])
            print(f"✅ Model loaded successfully!")
            
            return True
            
        except Exception as e:
            print(f"❌ Error loading model {model_name}: {str(e)}")
            print(f"🔄 Trying fallback model...")
            
            try:
                # Fallback to a reliable model
                self.pipeline = pipeline(
                    "sentiment-analysis",
                    model="distilbert-base-uncased-finetuned-sst-2-english",
                    device=-1,  # Use CPU as fallback
                    return_all_scores=True
                )
                print(f"✅ Fallback model loaded successfully!")
                return True
            except Exception as fallback_error:
                print(f"❌ Failed to load fallback model: {fallback_error}")
                return False
    
    def clean_text(self, text, max_length=128):
        """Clean and preprocess text"""
        if pd.isna(text) or not isinstance(text, str):
            return ""
        
        text = str(text).strip()
        text = ' '.join(text.split())
        
        if len(text) < 3:
            return ""
        
        if len(text) > max_length - 10:
            text = text[:max_length-13] + "..."
        
        return text
    
    def process_sentiment_result(self, result):
        """Process sentiment analysis result"""
        if isinstance(result, list):
            # Multi-label result
            sentiments = {item['label'].lower(): item['score'] for item in result}
            
            # Normalize label names
            label_mapping = {
                'positive': 'positive', 'pos': 'positive', 'label_2': 'positive',
                'negative': 'negative', 'neg': 'negative', 'label_0': 'negative',
                'neutral': 'neutral', 'neu': 'neutral', 'label_1': 'neutral'
            }
            
            normalized_sentiments = {}
            for label, score in sentiments.items():
                mapped_label = label_mapping.get(label, label)
                normalized_sentiments[mapped_label] = score
            
            dominant_sentiment = max(normalized_sentiments, key=normalized_sentiments.get)
            confidence = normalized_sentiments[dominant_sentiment]
            
            return {
                'sentiment': dominant_sentiment,
                'confidence': confidence,
                'positive': normalized_sentiments.get('positive', 0.0),
                'negative': normalized_sentiments.get('negative', 0.0),
                'neutral': normalized_sentiments.get('neutral', 0.0)
            }
        else:
            # Single result format
            sentiment = result['label'].lower()
            confidence = result['score']
            
            if sentiment in ['positive', 'pos']:
                return {
                    'sentiment': 'positive',
                    'confidence': confidence,
                    'positive': confidence,
                    'negative': (1 - confidence) / 2,
                    'neutral': (1 - confidence) / 2
                }
            elif sentiment in ['negative', 'neg']:
                return {
                    'sentiment': 'negative',
                    'confidence': confidence,
                    'positive': (1 - confidence) / 2,
                    'negative': confidence,
                    'neutral': (1 - confidence) / 2
                }
            else:
                return {
                    'sentiment': 'neutral',
                    'confidence': confidence,
                    'positive': (1 - confidence) / 2,
                    'negative': (1 - confidence) / 2,
                    'neutral': confidence
                }
    
    def analyze_batch(self, texts, batch_size=None):
        """Analyze sentiment for a batch of texts"""
        batch_size = batch_size or self.batch_size
        sentiment_results = []
        failed_count = 0
        
        print(f"🔄 Processing {len(texts)} texts in batches of {batch_size}")
        
        for i in tqdm(range(0, len(texts), batch_size), desc="Processing sentiment"):
            batch = texts[i:i + batch_size]
            
            # Clean texts
            cleaned_batch = []
            for text in batch:
                cleaned_text = self.clean_text(text)
                if cleaned_text:
                    cleaned_batch.append(cleaned_text)
                else:
                    sentiment_results.append({
                        'sentiment': 'neutral',
                        'confidence': 0.5,
                        'positive': 0.33,
                        'negative': 0.33,
                        'neutral': 0.34
                    })
                    failed_count += 1
            
            if not cleaned_batch:
                continue
            
            try:
                # Get sentiment predictions
                batch_results = self.pipeline(cleaned_batch)
                
                # Process results
                for result in batch_results:
                    sentiment_score = self.process_sentiment_result(result)
                    sentiment_results.append(sentiment_score)
                    
            except Exception as e:
                print(f"⚠️  Error processing batch {i//batch_size + 1}: {str(e)}")
                failed_count += len(cleaned_batch)
                
                # Add default neutral sentiment for failed batch
                for _ in cleaned_batch:
                    sentiment_results.append({
                        'sentiment': 'neutral',
                        'confidence': 0.5,
                        'positive': 0.33,
                        'negative': 0.33,
                        'neutral': 0.34
                    })
                
                # Clear GPU cache on error
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        
        if failed_count > 0:
            print(f"⚠️  Failed to process {failed_count} out of {len(texts)} texts")
        
        return sentiment_results

print("✅ BERTSentimentAnalyzer class created successfully!")

In [ ]:
# 📁 Local File Selection

print("📁 Select Your CSV File")
print("Required columns: 'comment_text', 'comment_owner_username'")
print("Optional columns: 'post_id', 'post_owner_username', 'comment_likes', 'post_likes'")

def select_csv_file():
    """Open file dialog to select CSV file"""
    try:
        # Hide the main tkinter window
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        
        # Open file dialog
        file_path = filedialog.askopenfilename(
            title="Select CSV File",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
            initialdir=os.getcwd()
        )
        
        root.destroy()
        return file_path
    except Exception as e:
        print(f"Error opening file dialog: {e}")
        return None

# Method 1: Interactive file selection
print("\n👆 Method 1: Click button to select file")

def load_file_button_clicked(b):
    global df
    file_path = select_csv_file()
    
    if file_path:
        try:
            df = pd.read_csv(file_path)
            print(f"\n✅ File loaded: {os.path.basename(file_path)}")
            print(f"📊 Dataset loaded successfully!")
            print(f"   Shape: {df.shape}")
            print(f"   Columns: {list(df.columns)}")
            
            # Check for required columns
            required_cols = ['comment_text', 'comment_owner_username']
            missing_cols = [col for col in required_cols if col not in df.columns]
            
            if missing_cols:
                print(f"❌ Missing required columns: {missing_cols}")
                print(f"💡 Please ensure your CSV has columns: {required_cols}")
                df = None
            else:
                print(f"✅ All required columns found!")
                
                # Show data preview
                print(f"\n📋 Data Preview:")
                display(df.head())
                
                # Show data summary
                valid_comments = df['comment_text'].dropna()
                print(f"\n📈 Data Summary:")
                print(f"   Total rows: {len(df):,}")
                print(f"   Valid comments: {len(valid_comments):,}")
                print(f"   Unique users: {df['comment_owner_username'].nunique():,}")
                
                if 'post_id' in df.columns:
                    print(f"   Unique posts: {df['post_id'].nunique():,}")
                    
        except Exception as e:
            print(f"❌ Error loading CSV file: {str(e)}")
            df = None
    else:
        print("❌ No file selected")
        df = None

# Create file selection button
file_button = widgets.Button(
    description='Select CSV File',
    button_style='primary',
    icon='folder-open'
)
file_button.on_click(load_file_button_clicked)
display(file_button)

# Method 2: Manual path entry
print("\n👆 Method 2: Enter file path manually")

def load_manual_path(file_path):
    global df
    
    if file_path and os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            print(f"\n✅ File loaded: {os.path.basename(file_path)}")
            print(f"📊 Dataset loaded successfully!")
            print(f"   Shape: {df.shape}")
            print(f"   Columns: {list(df.columns)}")
            
            # Check for required columns
            required_cols = ['comment_text', 'comment_owner_username']
            missing_cols = [col for col in required_cols if col not in df.columns]
            
            if missing_cols:
                print(f"❌ Missing required columns: {missing_cols}")
                print(f"💡 Please ensure your CSV has columns: {required_cols}")
                df = None
            else:
                print(f"✅ All required columns found!")
                
                # Show data preview
                print(f"\n📋 Data Preview:")
                display(df.head())
                
                # Show data summary
                valid_comments = df['comment_text'].dropna()
                print(f"\n📈 Data Summary:")
                print(f"   Total rows: {len(df):,}")
                print(f"   Valid comments: {len(valid_comments):,}")
                print(f"   Unique users: {df['comment_owner_username'].nunique():,}")
                
                if 'post_id' in df.columns:
                    print(f"   Unique posts: {df['post_id'].nunique():,}")
                    
        except Exception as e:
            print(f"❌ Error loading CSV file: {str(e)}")
            df = None
    else:
        print(f"❌ File not found: {file_path}")
        df = None

# Text widget for manual path entry
path_widget = widgets.Text(
    placeholder='Enter full path to CSV file (e.g., /path/to/your/file.csv)',
    description='File Path:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

load_button = widgets.Button(
    description='Load File',
    button_style='success'
)

def on_load_click(b):
    load_manual_path(path_widget.value)

load_button.on_click(on_load_click)

display(widgets.HBox([path_widget, load_button]))

# Initialize df as None
df = None

print("\n📝 Ready to load your CSV file using either method above!")

In [ ]:
# 🎛️ Model Configuration

if df is not None:
    print("🎛️ Configure Your Sentiment Analysis Model\n")
    
    # Model selection
    model_options = [
        "distilbert-base-uncased",
        "bert-base-uncased", 
        "roberta-base",
        "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "nlptown/bert-base-multilingual-uncased-sentiment"
    ]
    
    print("🤖 Available Models:")
    for i, model in enumerate(model_options, 1):
        print(f"   {i}. {model}")
    
    # Interactive model selection
    @interact
    def configure_model(
        model_choice=widgets.Dropdown(
            options=[(f"{i}. {model}", model) for i, model in enumerate(model_options, 1)],
            value=model_options[0],
            description='Model:'
        ),
        batch_size=widgets.IntSlider(
            value=16,
            min=4,
            max=64,
            step=4,
            description='Batch Size:'
        ),
        max_length=widgets.IntSlider(
            value=128,
            min=64,
            max=512,
            step=64,
            description='Max Length:'
        ),
        use_gpu=widgets.Checkbox(
            value=gpu_available,
            description='Use GPU',
            disabled=not gpu_available
        ),
        use_fine_tuning=widgets.Checkbox(
            value=False,
            description='Enable Fine-tuning'
        )
    ):
        global selected_model, selected_batch_size, selected_max_length, selected_use_gpu, selected_fine_tuning
        selected_model = model_choice
        selected_batch_size = batch_size
        selected_max_length = max_length
        selected_use_gpu = use_gpu
        selected_fine_tuning = use_fine_tuning
        
        print(f"\n📋 Current Configuration:")
        print(f"   Model: {selected_model}")
        print(f"   Batch Size: {selected_batch_size}")
        print(f"   Max Length: {selected_max_length}")
        print(f"   Use GPU: {selected_use_gpu}")
        print(f"   Fine-tuning: {selected_fine_tuning}")
        
        if selected_fine_tuning:
            print(f"   🔧 Fine-tuning hyperparameters: LR=2e-5, Epochs=3")
else:
    print("⚠️  Please upload a CSV file first!")

In [ ]:
# 🚀 Run Sentiment Analysis

if df is not None:
    print("🚀 Ready to Analyze Sentiment!")
    print(f"📊 Dataset: {len(df):,} comments")
    
    # Initialize analyzer
    analyzer = BERTSentimentAnalyzer(
        model_name=selected_model,
        batch_size=selected_batch_size,
        use_fine_tuning=selected_fine_tuning
    )
    
    # Load model
    if analyzer.load_model(selected_model, selected_use_gpu):
        print("\n🎯 Starting sentiment analysis...")
        start_time = time.time()
        
        # Prepare data
        valid_comments_df = df.dropna(subset=['comment_text']).copy()
        comment_texts = valid_comments_df['comment_text'].tolist()
        
        print(f"📝 Processing {len(comment_texts):,} valid comments")
        
        # Analyze sentiment
        sentiment_results = analyzer.analyze_batch(comment_texts, selected_batch_size)
        
        # Create results dataframe
        results_data = []
        for idx, (_, row) in enumerate(valid_comments_df.iterrows()):
            if idx < len(sentiment_results):
                sentiment_data = sentiment_results[idx]
                
                result_row = {
                    'comment_text': str(row['comment_text'])[:100] + '...' if len(str(row['comment_text'])) > 100 else str(row['comment_text']),
                    'comment_full_text': str(row['comment_text']),
                    'comment_owner_username': str(row.get('comment_owner_username', 'unknown')),
                    'post_owner_username': str(row.get('post_owner_username', 'unknown')),
                    'post_id': str(row.get('post_id', 'unknown')),
                    'sentiment': sentiment_data['sentiment'],
                    'confidence': sentiment_data['confidence'],
                    'positive': sentiment_data['positive'],
                    'negative': sentiment_data['negative'],
                    'neutral': sentiment_data['neutral'],
                    'comment_likes': int(row.get('comment_likes', 0)),
                    'post_likes': int(row.get('post_likes', 0)),
                    'engagement_rate': float(row.get('engagement_rate', 0.0))
                }
                results_data.append(result_row)
        
        sentiment_df = pd.DataFrame(results_data)
        
        # Calculate processing time
        end_time = time.time()
        processing_time = end_time - start_time
        
        print(f"\n✅ Sentiment analysis completed!")
        print(f"   📊 Analyzed: {len(sentiment_df):,} comments")
        print(f"   ⏱️  Processing time: {processing_time:.2f} seconds")
        print(f"   🚀 Speed: {len(sentiment_df)/processing_time:.1f} comments/second")
        
        # Show quick summary
        sentiment_counts = sentiment_df['sentiment'].value_counts()
        avg_confidence = sentiment_df['confidence'].mean()
        
        print(f"\n📈 Quick Summary:")
        print(f"   😊 Positive: {sentiment_counts.get('positive', 0):,} ({sentiment_counts.get('positive', 0)/len(sentiment_df)*100:.1f}%)")
        print(f"   😞 Negative: {sentiment_counts.get('negative', 0):,} ({sentiment_counts.get('negative', 0)/len(sentiment_df)*100:.1f}%)")
        print(f"   😐 Neutral: {sentiment_counts.get('neutral', 0):,} ({sentiment_counts.get('neutral', 0)/len(sentiment_df)*100:.1f}%)")
        print(f"   🎯 Average Confidence: {avg_confidence:.3f}")
        
    else:
        print("❌ Failed to load model. Please try again.")
        sentiment_df = None
else:
    print("⚠️  Please upload a CSV file first!")
    sentiment_df = None

In [ ]:
# 📊 Visualization Functions

def create_sentiment_distribution_chart(sentiment_df):
    """Create sentiment distribution pie chart"""
    sentiment_counts = sentiment_df['sentiment'].value_counts()
    
    fig = px.pie(
        values=sentiment_counts.values,
        names=sentiment_counts.index,
        title="📊 Sentiment Distribution",
        color_discrete_map={
            'positive': '#2E8B57',
            'negative': '#DC143C', 
            'neutral': '#4682B4'
        },
        hole=0.3
    )
    
    fig.update_traces(
        textposition='inside', 
        textinfo='percent+label',
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    )
    
    fig.update_layout(
        title_x=0.5,
        font=dict(size=14),
        showlegend=True,
        height=500
    )
    
    return fig

def create_confidence_distribution_chart(sentiment_df):
    """Create confidence score distribution"""
    fig = px.histogram(
        sentiment_df, 
        x='confidence',
        color='sentiment',
        title="🎯 Confidence Score Distribution",
        nbins=20,
        color_discrete_map={
            'positive': '#2E8B57',
            'negative': '#DC143C', 
            'neutral': '#4682B4'
        }
    )
    
    fig.update_layout(
        title_x=0.5,
        xaxis_title="Confidence Score",
        yaxis_title="Count",
        font=dict(size=12),
        height=500
    )
    
    return fig

def create_user_sentiment_analysis(sentiment_df, top_n=15):
    """Analyze sentiment by user"""
    user_stats = sentiment_df.groupby('comment_owner_username').agg({
        'sentiment': lambda x: x.value_counts().to_dict(),
        'confidence': 'mean',
        'comment_text': 'count'
    }).round(3)
    
    user_stats.columns = ['Sentiment_Distribution', 'Avg_Confidence', 'Total_Comments']
    user_stats['Positive_Ratio'] = user_stats['Sentiment_Distribution'].apply(
        lambda x: x.get('positive', 0) / sum(x.values()) if sum(x.values()) > 0 else 0
    )
    
    top_users = user_stats.nlargest(top_n, 'Total_Comments')
    
    # Create subplot
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Top Users by Comment Count',
            'User Sentiment Positivity',
            'User Activity vs Confidence',
            'Sentiment Distribution by Top Users'
        ],
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "scatter"}, {"type": "bar"}]]
    )
    
    # Top users by comment count
    fig.add_trace(
        go.Bar(
            x=top_users.index[:10],
            y=top_users['Total_Comments'][:10],
            name='Comments',
            marker_color='#1f77b4'
        ),
        row=1, col=1
    )
    
    # User positivity
    positive_users = top_users.nlargest(10, 'Positive_Ratio')
    fig.add_trace(
        go.Bar(
            x=positive_users.index,
            y=positive_users['Positive_Ratio'],
            name='Positive Ratio',
            marker_color='#2E8B57'
        ),
        row=1, col=2
    )
    
    # Activity vs confidence scatter
    active_users = user_stats[user_stats['Total_Comments'] >= 2].head(20)
    fig.add_trace(
        go.Scatter(
            x=active_users['Total_Comments'],
            y=active_users['Avg_Confidence'],
            mode='markers',
            marker=dict(
                size=8,
                color=active_users['Positive_Ratio'],
                colorscale='RdYlGn',
                showscale=True,
                colorbar=dict(title="Positive Ratio")
            ),
            text=active_users.index,
            hovertemplate='<b>%{text}</b><br>Comments: %{x}<br>Confidence: %{y}<extra></extra>',
            name='Users'
        ),
        row=2, col=1
    )
    
    # Sentiment distribution for top users
    user_sentiment_data = []
    for user, row in top_users.head(8).iterrows():
        sentiment_dist = row['Sentiment_Distribution']
        for sentiment, count in sentiment_dist.items():
            user_sentiment_data.append({
                'User': user[:15] + '...' if len(user) > 15 else user,
                'Sentiment': sentiment,
                'Count': count
            })
    
    if user_sentiment_data:
        user_sentiment_df = pd.DataFrame(user_sentiment_data)
        for sentiment in ['positive', 'negative', 'neutral']:
            sentiment_data = user_sentiment_df[user_sentiment_df['Sentiment'] == sentiment]
            if not sentiment_data.empty:
                color_map = {'positive': '#2E8B57', 'negative': '#DC143C', 'neutral': '#4682B4'}
                fig.add_trace(
                    go.Bar(
                        x=sentiment_data['User'],
                        y=sentiment_data['Count'],
                        name=sentiment.title(),
                        marker_color=color_map[sentiment]
                    ),
                    row=2, col=2
                )
    
    fig.update_layout(
        title_text="👥 User Sentiment Analysis",
        title_x=0.5,
        height=800,
        showlegend=True
    )
    
    # Update x-axis labels
    fig.update_xaxes(tickangle=45, row=1, col=1)
    fig.update_xaxes(tickangle=45, row=1, col=2)
    fig.update_xaxes(tickangle=45, row=2, col=2)
    
    return fig

def create_sample_comments_display(sentiment_df, samples_per_sentiment=3):
    """Display sample comments for each sentiment"""
    html_content = "<div style='font-family: Arial, sans-serif;'>"
    html_content += "<h3>📝 Sample Comments by Sentiment</h3>"
    
    sentiment_categories = ['positive', 'negative', 'neutral']
    emoji_map = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}
    color_map = {'positive': '#2E8B57', 'negative': '#DC143C', 'neutral': '#4682B4'}
    
    for sentiment in sentiment_categories:
        sentiment_data = sentiment_df[sentiment_df['sentiment'] == sentiment]
        
        if not sentiment_data.empty:
            top_samples = sentiment_data.nlargest(samples_per_sentiment, 'confidence')
            
            html_content += f"<h4>{emoji_map[sentiment]} {sentiment.title()} Comments</h4>"
            
            for idx, row in top_samples.iterrows():
                comment_text = row.get('comment_full_text', row.get('comment_text', 'No text available'))
                confidence = row.get('confidence', 0.0)
                username = row.get('comment_owner_username', 'Unknown')
                
                display_text = comment_text[:200] + "..." if len(str(comment_text)) > 200 else comment_text
                
                html_content += f"""
                <div style="border-left: 4px solid {color_map[sentiment]}; padding: 10px; margin: 10px 0; background-color: #f8f9fa; border-radius: 5px;">
                    <strong>@{username}</strong> <span style="color: #666;">(Confidence: {confidence:.3f})</span><br>
                    <em>"{display_text}"</em>
                </div>
                """
        else:
            html_content += f"<h4>{emoji_map[sentiment]} {sentiment.title()} Comments</h4>"
            html_content += "<p style='color: #666;'>No comments found for this sentiment.</p>"
    
    html_content += "</div>"
    return HTML(html_content)

print("📊 Visualization functions created successfully!")

In [ ]:
# 📈 Comprehensive Analysis Results

if sentiment_df is not None and len(sentiment_df) > 0:
    print("📊 Generating Comprehensive Analysis...\n")
    
    # Summary Statistics
    print("📈 Summary Statistics")
    print("=" * 50)
    
    total_comments = len(sentiment_df)
    avg_confidence = sentiment_df['confidence'].mean()
    sentiment_counts = sentiment_df['sentiment'].value_counts()
    high_confidence = (sentiment_df['confidence'] > 0.8).sum()
    
    print(f"📊 Total Comments Analyzed: {total_comments:,}")
    print(f"🎯 Average Confidence: {avg_confidence:.3f}")
    print(f"🔥 High Confidence (>0.8): {high_confidence:,} ({high_confidence/total_comments*100:.1f}%)")
    print(f"\n💭 Sentiment Breakdown:")
    for sentiment, count in sentiment_counts.items():
        percentage = count/total_comments*100
        emoji = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}[sentiment]
        print(f"   {emoji} {sentiment.title()}: {count:,} ({percentage:.1f}%)")
    
    # Display visualizations
    print("\n" + "=" * 50)
    print("📊 VISUALIZATION DASHBOARD")
    print("=" * 50)
    
    # 1. Sentiment Distribution
    print("\n1️⃣ Sentiment Distribution")
    fig1 = create_sentiment_distribution_chart(sentiment_df)
    fig1.show()
    
    # 2. Confidence Distribution
    print("\n2️⃣ Confidence Score Analysis")
    fig2 = create_confidence_distribution_chart(sentiment_df)
    fig2.show()
    
    # 3. Average sentiment scores
    print("\n3️⃣ Average Sentiment Scores")
    avg_scores = sentiment_df[['positive', 'negative', 'neutral']].mean()
    fig3 = px.bar(
        x=avg_scores.index,
        y=avg_scores.values,
        title="📊 Average Sentiment Scores",
        color=avg_scores.index,
        color_discrete_map={
            'positive': '#2E8B57',
            'negative': '#DC143C', 
            'neutral': '#4682B4'
        },
        text=avg_scores.values.round(3)
    )
    fig3.update_traces(texttemplate='%{text}', textposition='outside')
    fig3.update_layout(
        title_x=0.5,
        yaxis_title="Average Score",
        font=dict(size=12),
        height=400
    )
    fig3.show()
    
    # 4. User Analysis (if we have enough users)
    unique_users = sentiment_df['comment_owner_username'].nunique()
    if unique_users > 5:
        print(f"\n4️⃣ User Sentiment Analysis ({unique_users:,} unique users)")
        fig4 = create_user_sentiment_analysis(sentiment_df)
        fig4.show()
    else:
        print(f"\n4️⃣ User Analysis skipped (only {unique_users} unique users)")
    
    # 5. Confidence vs Sentiment relationship
    print("\n5️⃣ Confidence vs Sentiment Analysis")
    conf_stats = sentiment_df.groupby('sentiment')['confidence'].agg(['mean', 'std', 'count']).round(3)
    
    fig5 = go.Figure()
    
    for sentiment in ['positive', 'negative', 'neutral']:
        if sentiment in conf_stats.index:
            sentiment_data = sentiment_df[sentiment_df['sentiment'] == sentiment]
            color_map = {'positive': '#2E8B57', 'negative': '#DC143C', 'neutral': '#4682B4'}
            
            fig5.add_trace(go.Box(
                y=sentiment_data['confidence'],
                name=sentiment.title(),
                marker_color=color_map[sentiment],
                boxpoints='outliers'
            ))
    
    fig5.update_layout(
        title="🎯 Confidence Score Distribution by Sentiment",
        title_x=0.5,
        yaxis_title="Confidence Score",
        xaxis_title="Sentiment",
        font=dict(size=12),
        height=400
    )
    fig5.show()
    
    # 6. Show confidence statistics table
    print("\n📊 Confidence Statistics by Sentiment:")
    display(conf_stats)
    
    # 7. Sample Comments
    print("\n6️⃣ Sample Comments by Sentiment")
    sample_comments = create_sample_comments_display(sentiment_df, samples_per_sentiment=3)
    display(sample_comments)
    
else:
    print("⚠️  No sentiment analysis results available. Please run the analysis first!")

In [ ]:
# 💾 Export Results to Local Files

if sentiment_df is not None and len(sentiment_df) > 0:
    print("💾 Export Your Results to Local Files")
    print("=" * 40)
    
    # Prepare export data
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Create comprehensive results
    export_data = {
        'sentiment_scores': sentiment_df.to_dict('records'),
        'metadata': {
            'total_comments_analyzed': len(sentiment_df),
            'model_used': selected_model,
            'batch_size': selected_batch_size,
            'max_length': selected_max_length,
            'analysis_timestamp': datetime.now().isoformat(),
            'gpu_used': selected_use_gpu,
            'fine_tuning_enabled': selected_fine_tuning
        },
        'summary_statistics': {
            'sentiment_distribution': sentiment_df['sentiment'].value_counts().to_dict(),
            'average_confidence': float(sentiment_df['confidence'].mean()),
            'confidence_std': float(sentiment_df['confidence'].std()),
            'high_confidence_count': int((sentiment_df['confidence'] > 0.8).sum()),
            'low_confidence_count': int((sentiment_df['confidence'] < 0.6).sum())
        }
    }
    
    # Export to local files
    print("\n📁 Saving files to local directory:")
    
    # 1. CSV Export
    print("\n1️⃣ CSV Export")
    csv_filename = output_dir / f"sentiment_analysis_results_{timestamp}.csv"
    sentiment_df.to_csv(csv_filename, index=False)
    print(f"✅ CSV file saved: {csv_filename}")
    
    # 2. JSON Export (Full Results)
    print("\n2️⃣ JSON Export (Full Results)")
    json_filename = output_dir / f"sentiment_analysis_full_{timestamp}.json"
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, default=str, ensure_ascii=False)
    print(f"✅ JSON file saved: {json_filename}")
    
    # 3. Summary Report
    print("\n3️⃣ Summary Report")
    summary_filename = output_dir / f"sentiment_summary_{timestamp}.txt"
    
    with open(summary_filename, 'w', encoding='utf-8') as f:
        f.write("SENTIMENT ANALYSIS SUMMARY REPORT\n")
        f.write("=" * 40 + "\n\n")
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Model Used: {selected_model}\n")
        f.write(f"Total Comments: {len(sentiment_df):,}\n")
        f.write(f"Average Confidence: {sentiment_df['confidence'].mean():.3f}\n\n")
        
        f.write("SENTIMENT DISTRIBUTION:\n")
        f.write("-" * 25 + "\n")
        sentiment_counts = sentiment_df['sentiment'].value_counts()
        for sentiment, count in sentiment_counts.items():
            percentage = count/len(sentiment_df)*100
            f.write(f"{sentiment.title()}: {count:,} ({percentage:.1f}%)\n")
        
        f.write("\nCONFIDENCE STATISTICS:\n")
        f.write("-" * 22 + "\n")
        conf_stats = sentiment_df.groupby('sentiment')['confidence'].agg(['mean', 'std', 'count'])
        for sentiment in conf_stats.index:
            stats = conf_stats.loc[sentiment]
            f.write(f"{sentiment.title()}: Mean={stats['mean']:.3f}, Std={stats['std']:.3f}, Count={stats['count']}\n")
        
        f.write("\nTOP POSITIVE COMMENTS:\n")
        f.write("-" * 22 + "\n")
        positive_comments = sentiment_df[sentiment_df['sentiment'] == 'positive'].nlargest(3, 'confidence')
        for idx, row in positive_comments.iterrows():
            comment = row['comment_full_text'][:100] + "..." if len(row['comment_full_text']) > 100 else row['comment_full_text']
            f.write(f"• {comment} (Confidence: {row['confidence']:.3f})\n")
        
        f.write("\nTOP NEGATIVE COMMENTS:\n")
        f.write("-" * 22 + "\n")
        negative_comments = sentiment_df[sentiment_df['sentiment'] == 'negative'].nlargest(3, 'confidence')
        for idx, row in negative_comments.iterrows():
            comment = row['comment_full_text'][:100] + "..." if len(row['comment_full_text']) > 100 else row['comment_full_text']
            f.write(f"• {comment} (Confidence: {row['confidence']:.3f})\n")
    
    print(f"✅ Summary report saved: {summary_filename}")
    
    # 4. Excel Export (if openpyxl is available)
    try:
        import openpyxl
        print("\n4️⃣ Excel Export")
        excel_filename = output_dir / f"sentiment_analysis_results_{timestamp}.xlsx"
        
        with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
            sentiment_df.to_excel(writer, sheet_name='Sentiment_Results', index=False)
            
            # Summary sheet
            summary_df = pd.DataFrame({
                'Metric': ['Total Comments', 'Average Confidence', 'Positive Count', 'Negative Count', 'Neutral Count'],
                'Value': [
                    len(sentiment_df),
                    f"{sentiment_df['confidence'].mean():.3f}",
                    sentiment_counts.get('positive', 0),
                    sentiment_counts.get('negative', 0),
                    sentiment_counts.get('neutral', 0)
                ]
            })
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
            
        print(f"✅ Excel file saved: {excel_filename}")
    except ImportError:
        print("\n4️⃣ Excel Export (skipped - openpyxl not installed)")
        print("   Install with: pip install openpyxl")
    
    print("\n✅ All files exported successfully!")
    print(f"📁 Files saved to: {output_dir.absolute()}")
    print(f"   • {csv_filename.name} - Detailed results in CSV format")
    print(f"   • {json_filename.name} - Complete analysis data in JSON format")
    print(f"   • {summary_filename.name} - Human-readable summary report")
    
    # Create download links for Jupyter
    if jupyter_env:
        print("\n🔗 Download Links:")
        display(FileLink(str(csv_filename), result_html_prefix="CSV: "))
        display(FileLink(str(json_filename), result_html_prefix="JSON: "))
        display(FileLink(str(summary_filename), result_html_prefix="Summary: "))
        if 'excel_filename' in locals():
            display(FileLink(str(excel_filename), result_html_prefix="Excel: "))
    
else:
    print("⚠️  No results to export. Please run sentiment analysis first!")

In [ ]:
# 📊 Performance Metrics & Memory Usage

if sentiment_df is not None:
    print("📊 Performance Metrics & System Information")
    print("=" * 50)
    
    # GPU Information
    if torch.cuda.is_available():
        print("🔥 GPU Information:")
        print(f"   Device: {torch.cuda.get_device_name(0)}")
        print(f"   Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
        print(f"   Memory Cached: {torch.cuda.memory_reserved(0) / 1024**2:.1f} MB")
        print(f"   Memory Usage: {torch.cuda.memory_allocated(0) / torch.cuda.get_device_properties(0).total_memory * 100:.1f}%")
    else:
        print("💻 Using CPU for processing")
    
    # Model Information
    print(f"\n🤖 Model Information:")
    print(f"   Model: {selected_model}")
    print(f"   Batch Size: {selected_batch_size}")
    print(f"   Max Length: {selected_max_length}")
    print(f"   Fine-tuning: {selected_fine_tuning}")
    
    # Analysis Performance
    if 'processing_time' in locals():
        print(f"\n⚡ Performance Metrics:")
        print(f"   Total Processing Time: {processing_time:.2f} seconds")
        print(f"   Comments per Second: {len(sentiment_df)/processing_time:.1f}")
        print(f"   Average Time per Comment: {processing_time/len(sentiment_df)*1000:.2f} ms")
    
    # Dataset Statistics
    print(f"\n📊 Dataset Statistics:")
    print(f"   Total Comments Processed: {len(sentiment_df):,}")
    print(f"   Unique Users: {sentiment_df['comment_owner_username'].nunique():,}")
    if 'post_id' in sentiment_df.columns:
        print(f"   Unique Posts: {sentiment_df['post_id'].nunique():,}")
    
    # Memory cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\n🧹 GPU memory cleared")
    
print("\n✅ Analysis Complete!")
print("🎉 Thank you for using the BERT Sentiment Analysis notebook!")
print("\n💡 Tips for future use:")
print("   • Use GPU for faster processing on large datasets")
print("   • Experiment with different BERT models for better accuracy")
print("   • Consider fine-tuning for domain-specific analysis")
print("   • Export results in multiple formats for further analysis")

## 🛠️ Troubleshooting & FAQ for Local PC

### Common Issues & Solutions

#### 🚫 GPU Not Available
**Problem**: "GPU not available, using CPU"
**Solutions**: 
1. Install CUDA-compatible PyTorch:
   ```bash
   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
   ```
2. Check NVIDIA drivers are installed and up to date
3. Verify CUDA installation: `nvidia-smi`
4. For AMD GPUs, use ROCm version of PyTorch

#### 📁 File Selection Issues
**Problem**: File dialog doesn't open or CSV won't load
**Solutions**:
- Ensure file is in CSV format with proper encoding (UTF-8)
- Check that required columns exist: `comment_text`, `comment_owner_username`
- Use the manual path entry method if dialog fails
- File size should be reasonable (<500MB for smooth processing)
- Check file permissions (read access required)

#### 🤖 Model Loading Errors
**Problem**: Model fails to load
**Solutions**:
- Check internet connection (models download from Hugging Face)
- Try a different model (DistilBERT is most reliable)
- Clear cache: `rm -rf ~/.cache/huggingface/`
- Use CPU mode if GPU fails
- Check available disk space (models can be 500MB+)

#### 💾 Memory Issues
**Problem**: "CUDA out of memory" or system freezes
**Solutions**:
- Reduce batch size (try 8, 4, or even 2)
- Use a smaller model (DistilBERT instead of BERT)
- Close other applications using GPU/RAM
- Process data in smaller chunks
- Clear GPU memory: `torch.cuda.empty_cache()`
- Monitor system resources during processing

#### 📝 Jupyter/IPython Issues
**Problem**: Widgets not working or kernel crashes
**Solutions**:
- Install/enable widget extensions:
  ```bash
  pip install ipywidgets
  jupyter nbextension enable --py widgetsnbextension
  ```
- Restart Jupyter kernel
- Use JupyterLab instead of Jupyter Notebook
- Update Jupyter: `pip install --upgrade jupyter`

### Performance Tips for Local PC

1. **For Large Datasets (>10K comments)**:
   - Use GPU with larger batch sizes (32-64) if you have enough VRAM
   - Monitor RAM usage - close unnecessary applications
   - Consider processing in chunks for very large datasets
   - Use SSD storage for faster file I/O

2. **For Better Accuracy**:
   - Use domain-specific models (e.g., Twitter models for social media)
   - Enable fine-tuning for your specific use case
   - Manually review high-confidence predictions
   - Consider ensemble methods with multiple models

3. **For Faster Processing**:
   - Use DistilBERT (3x faster than BERT, 97% accuracy)
   - Increase batch size within memory limits
   - Reduce max_length for shorter texts
   - Use FP16 precision if supported
   - Close browser tabs and other GPU-using applications

### Model Recommendations for Local PC

| Use Case | Recommended Model | RAM/VRAM Requirements |
|----------|-------------------|----------------------|
| General Text | `distilbert-base-uncased` | 2GB RAM, 1GB VRAM |
| Social Media | `cardiffnlp/twitter-roberta-base-sentiment-latest` | 3GB RAM, 2GB VRAM |
| Multilingual | `nlptown/bert-base-multilingual-uncased-sentiment` | 4GB RAM, 2GB VRAM |
| High Accuracy | `roberta-base` | 4GB RAM, 3GB VRAM |
| Fast Processing | `distilbert-base-uncased` | 2GB RAM, 1GB VRAM |

### Local Environment Setup

#### **Python Environment**:
```bash
# Create virtual environment
python -m venv sentiment_env

# Activate (Linux/Mac)
source sentiment_env/bin/activate

# Activate (Windows)
sentiment_env\Scripts\activate

# Install packages
pip install -r requirements.txt
```

#### **GPU Setup (NVIDIA)**:
```bash
# Check CUDA version
nvidia-smi

# Install appropriate PyTorch
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Verify installation
python -c "import torch; print(torch.cuda.is_available())"
```

### Data Format Requirements

**Required Columns**:
- `comment_text`: The text to analyze
- `comment_owner_username`: Username of commenter

**Optional Columns**:
- `post_id`: Identifier for the post
- `post_owner_username`: Username of post owner
- `comment_likes`: Number of likes on comment
- `post_likes`: Number of likes on post
- `engagement_rate`: Engagement metrics

**Example CSV Structure**:
```csv
comment_text,comment_owner_username,post_id,post_owner_username
"This is amazing!",user123,post_001,creator456
"Not so good",user789,post_001,creator456
"Love this content",user456,post_002,creator789
```

### Output Files Location

All results are saved to the `sentiment_analysis_output/` directory in your notebook location:
- CSV files for spreadsheet analysis
- JSON files for programmatic access
- TXT files for human-readable reports
- Excel files (if openpyxl installed)

---

## 🎯 Next Steps

After completing the analysis on your local PC:

1. **📊 Further Analysis**: Use exported files in Excel, Tableau, R, or other tools
2. **🔄 Batch Processing**: Process multiple CSV files programmatically
3. **🎯 Custom Analysis**: Modify code for specific research questions
4. **📈 Advanced Visualization**: Create custom charts using exported data
5. **🤖 Automation**: Create scripts for regular sentiment monitoring
6. **🔄 Model Comparison**: Run same data through different models
7. **📊 Statistical Analysis**: Perform significance tests on results

---

**Optimized for local PC environments with full control over your data and processing 💻❤️**